# NB02 – Daten Bereinigung

**Grid-Arbitrage** · Batteriespeicher-Arbitrage im Schweizer Strommarkt

**Gruppe:** SC26_Gruppe_2 | **Verantwortlich:** Senthuran Elankeswaran | **Datum:** März 2026

---

*Bereinigt Spot-Preise: clippt Ausreisser, füllt Lücken, speichert in `data/processed/`.*


| [← NB01 Daten Laden](01_Daten_Laden.ipynb) | [↑ Übersicht ↑](../organisation/O_01_Project_Overview.ipynb) | [NB03 Daten Analyse →](03_Daten_Analyse.ipynb) |
|:---|:---:|---:|

## Inhaltsverzeichnis<a id='toc_NB_02'></a>

[Einleitung](#einleitung_NB_02)  
[Initialisierung](#initialisierung_NB_02)  
1 [Datenbereinigung Spot-Preise](#datenbereinigung-spot-preise_NB_02)  
[Fazit](#fazit_NB_02)  
[Abschluss](#abschluss_NB_02)  


---
## Einleitung <a id='einleitung_NB_02'></a>

[↑ Inhaltsverzeichnis](#toc_NB_02)

Bereinigung der ENTSO-E-Rohdaten aus NB01:

- **UTC-Normierung** — alle Zeitstempel auf UTC setzen (vermeidet Probleme durch
  Sommer-/Winterzeit-Umstellung und doppelte/fehlende Stunden)
- **Deduplizierung** — doppelte Stunden bei der Zeitumstellung Oktober entfernen
- **Kurzlücken schliessen** — Einzelwerte (≤3h) per linearer Interpolation,
  längere Lücken (≤6h) per Forward/Backward-Fill als Fallback,
  Lücken >6h bleiben als `NaN` für Nachvollziehbarkeit

Output: `data/processed/spot_prices_clean.csv` — Grundlage für Dispatch- und
Wirtschaftlichkeitsrechnung in NB03.


## Initialisierung<a id='initialisierung_NB_02'></a>

[↑ Inhaltsverzeichnis](#toc_NB_02)

Lädt `../sync/config.json` (SSOT), setzt Verzeichnispfade, liest Simulation- und Wirtschaftlichkeitsparameter sowie Datenzeitraum aus `../sync/transfer.json`.

**Setup – Konfiguration & Verzeichnisstruktur:** Lädt `../sync/config.json`, setzt Pfade
und liest `FORCE_RELOAD`, `LIFETIME` und `ZIEL_ROI` als lokale Aliases.


In [1]:
# ── lib/ aus Projekt-Root erreichbar machen + lib-Imports ───────────────────
# Notebook liegt in einem Unterordner (kuer/, experimental/, notebooks/,
# organisation/). Damit 'from lib.xxx import ...' funktioniert, muss der
# Projekt-Root vorne in sys.path stehen. autoreload sorgt dafür, dass
# Änderungen in lib/*.py ohne Kernel-Restart übernommen werden.
import sys, os
_PROJECT_ROOT = os.path.abspath('..')
if _PROJECT_ROOT not in sys.path:
    sys.path.insert(0, _PROJECT_ROOT)
try:
    get_ipython().run_line_magic('load_ext', 'autoreload')
    get_ipython().run_line_magic('autoreload', '2')
except Exception:
    pass

# lib-Imports (einmal zentral — in allen folgenden Zellen verfügbar)
from lib.plotting import show_source
from lib.io_ops   import load_transfer, save_transfer, log_dataindex, needs_rebuild, log_missing, final_check

print(f'lib-Pfad aktiv: {_PROJECT_ROOT}/lib')


lib-Pfad aktiv: C:\Users\patri\Documents\docker\python\OWN\Grid-Arbitrage/lib


In [2]:
import os, sys, warnings, json
import numpy  as np
import pandas as pd
from datetime import datetime
warnings.filterwarnings('ignore')

# Versionen anzeigen für Reproduzierbarkeit
print(f"Numpy        Version: {np.__version__}")
print(f"Pandas       Version: {pd.__version__}")
print(f"📅 Zuletzt ausgeführt am: {datetime.now().strftime('%d.%m.%Y um %H:%M:%S')}")

Numpy        Version: 2.2.6
Pandas       Version: 2.3.3
📅 Zuletzt ausgeführt am: 26.04.2026 um 12:51:48


**🔎 Quellcode der importierten lib-Funktion**

Die Funktion `load_transfer` wird aus `lib/io_ops.py` importiert und
liest Einträge aus `sync/transfer.json`. Aufklappbar ist der Quellcode einsehbar.


In [3]:
show_source(load_transfer)


<details>
<summary>🔎 Quellcode: <code>load_transfer</code> (aus <code>lib/io_ops.py</code>)</summary>

```python
def load_transfer(path='../sync/transfer.json', key=None, default=None):
    """Lädt transfer.json und gibt das ganze Dict oder einen Teil zurück.

    Verhalten
    ---------
    * Datei existiert nicht oder ist leer → Rückgabe ist ``default`` (bei
      key=None: ``default`` oder ``{}``). Gibt Warnung auf stdout aus.
    * Datei existiert → gibt bei ``key=None`` das ganze Dict zurück, bei
      gegebenem ``key`` nur den entsprechenden Teilbaum (``default``, wenn
      Key fehlt).

    Parameter
    ---------
    path : str, default '../sync/transfer.json'
        Pfad zur transfer.json.
    key : str, optional
        Top-Level-Key ('datenzeitraum', 'simulation', ...). Bei None wird
        das komplette Dict zurückgegeben.
    default : any, optional
        Rückgabewert bei fehlender Datei oder fehlendem Key. Bei key=None
        ist der Default ``{}``.

    Return
    ------
    dict oder der Wert des angefragten Keys.
    """
    import json as _json

    if default is None and key is None:
        default = {}

    if not os.path.exists(path) or os.path.getsize(path) == 0:
        print(f'⚠️  {path} nicht gefunden oder leer — NB01/NB02 zuerst ausführen')
        return default

    with open(path, encoding='utf-8') as _f:
        data = _json.load(_f)

    if key is None:
        return data
    return data.get(key, default)
```

</details>


In [4]:
# ── ../sync/config.json laden (Single Source of Truth) ───────────────────────────────
# NEVER hardcode MODE oder FORCE_RELOAD hier — immer aus ../sync/config.json lesen
with open('../sync/config.json') as _f:
    CFG = json.load(_f)

MODE         = CFG['mode']
FORCE_RELOAD = CFG['force_reload']

# ── Sim-Parameter aus ../sync/config.json ────────────────────────────────────────────
_sim         = CFG['pflicht']['simulation']
CHARGE_Q     = _sim['charge_quantile']      # 0.25
DISCHARGE_Q  = _sim['discharge_quantile']   # 0.75
SOC_MIN_PCT  = _sim['soc_min_pct']          # 0.05
SOC_MAX_PCT  = _sim['soc_max_pct']          # 0.95
EFFICIENCY   = _sim['efficiency_roundtrip'] # 0.92

# ── Wirtschaftlichkeits-Parameter ─────────────────────────────────────────────
_w           = CFG['pflicht']['wirtschaftlichkeit']
CAPEX_EUR_KWH = _w['capex_eur_kwh']
OPEX_RATE    = _w['opex_rate']
LIFETIME     = _w['lifetime_j']
ZIEL_ROI_PCT = round(100 / LIFETIME, 2)  # Ziel-ROI = 1/lifetime_j: ROI der nötig ist, um BE in LIFETIME Jahren zu erreichen

# ── Szenario-Parameter (Gleichzeitigkeit) ─────────────────────────────────────
_sz           = CFG['szenarien']
GLEICHZEITIGKEIT = _sz['gleichzeitigkeit_aktiv']          # 'pessimistisch'|'realistisch'|'optimistisch'
RATE          = _sz['optionen'][GLEICHZEITIGKEIT]['rate']
SZ_OPT        = _sz['optionen'][GLEICHZEITIGKEIT]         # alle Parameter des aktiven Szenarios
CH_SPITZENLAST_GW = _sz['ch_spitzenlast_gw']


In [5]:
DIR_RAW          = os.path.join('../data', 'raw')
DIR_PROCESSED    = os.path.join('../data', 'processed')
# intermediate: Szenario-abhängige Dateien in Unterordner
DIR_INTER_BASE   = os.path.join('../data', 'intermediate')
DIR_INTER_SZ     = os.path.join(DIR_INTER_BASE, GLEICHZEITIGKEIT)   # z.B. data/intermediate/realistisch/
DIR_INTER        = DIR_INTER_BASE  # szenario-unabhängige Dateien (spread, import/export)
DATAINDEX        = '../sync/dataindex.csv'

for d in [DIR_RAW, DIR_PROCESSED, DIR_INTER_BASE, DIR_INTER_SZ]:
    os.makedirs(d, exist_ok=True)

print(f'../sync/config.json  : geladen')
print(f'MODE         : {MODE}')
print(f'Gleichz.     : {GLEICHZEITIGKEIT} ({RATE*100:.0f}%)')
print(f'raw          : {os.path.abspath(DIR_RAW)}')
print(f'processed    : {os.path.abspath(DIR_PROCESSED)}')
print(f'intermediate : {os.path.abspath(DIR_INTER_SZ)} (szenario-abhängig)')


../sync/config.json  : geladen
MODE         : data
Gleichz.     : realistisch (40%)
raw          : C:\Users\patri\Documents\docker\python\OWN\Grid-Arbitrage\data\raw
processed    : C:\Users\patri\Documents\docker\python\OWN\Grid-Arbitrage\data\processed
intermediate : C:\Users\patri\Documents\docker\python\OWN\Grid-Arbitrage\data\intermediate\realistisch (szenario-abhängig)


In [6]:
# -- Transfer: Ergebnisse aus ../sync/transfer.json laden ----------------------------
TF       = load_transfer()
_dt      = TF.get('datenzeitraum', {})
_sim_tf  = TF.get('simulation', {})
TF_N_YEARS = _dt.get('n_years', None)
TF_START   = _dt.get('start_date', 'unbekannt')
TF_END     = _dt.get('end_date',   'unbekannt')
TF_SPREAD  = _sim_tf.get('spread_mean_eur_mwh', None)
TF_ECON    = _sim_tf.get('wirtschaftlichkeit', {})
TF_HYB     = TF.get('hybrid_simulation', {}).get('ergebnisse', {})
TF_KUER    = CFG.get('kuer_aktiv', {})
if TF:
    print(f"../sync/transfer.json: {TF_START} – {TF_END} ({TF_N_YEARS} Jahre) | Spread: {TF_SPREAD} EUR/MWh")


../sync/transfer.json: 2023 – 2026 (3.32 Jahre) | Spread: None EUR/MWh


**Transfer NB01 → NB02:** Datenzeitraum (`n_years`) aus `../sync/transfer.json` lesen —
SSOT für alle Jahresdurchschnitts-Berechnungen. Fehlt die Datei: Fallback auf Selbstberechnung.


In [7]:
# -- Transfer: n_years von NB01 übernehmen (SSOT: gleiche Methode überall) ----
_dt_r  = load_transfer(key='datenzeitraum', default={})
n_years = _dt_r.get('n_years', None)
_sz_tf  = _dt_r.get('_szenario_aktiv', None)
if n_years:
    print(f'n_years aus ../sync/transfer.json (NB01): {n_years:.3f}')
if _sz_tf and _sz_tf != SZ_AKTIV:
    print(f'⚠️  Szenario-Konflikt: transfer={_sz_tf} vs config={SZ_AKTIV}')
    print('   NB01 mit aktuellem Szenario neu ausführen!')


n_years aus ../sync/transfer.json (NB01): 3.320


**Hilfsfunktionen:** `log_dataindex()` für das Datenprotokoll — aus `lib.io_ops` importiert
(gleicher Aufruf wie in NB01).


**🔎 Quellcode der importierten lib-Funktion**

Die Funktion `log_dataindex` wird aus `lib/io_ops.py` importiert und
schreibt Einträge ins Daten-Provenienz-Protokoll `sync/dataindex.csv`.
Bereits bestehende Einträge für denselben Dateinamen werden als
`superseded` markiert. Aufklappbar darunter ist der Quellcode einsehbar.


In [8]:
show_source(log_dataindex)


<details>
<summary>🔎 Quellcode: <code>log_dataindex</code> (aus <code>lib/io_ops.py</code>)</summary>

```python
def log_dataindex(filename, source_url, local_path, data_type,
                  rows=None, size_kb=None, status='active', note='',
                  dataindex_path=None):
    """Schreibt einen Eintrag ins Daten-Provenienz-Protokoll.

    Existiert bereits ein aktiver Eintrag mit demselben ``filename``, wird
    dieser als ``superseded`` markiert (mit Zeitstempel in ``superseded_at``).

    Parameter
    ---------
    filename : str
        Dateiname (ohne Pfad).
    source_url : str
        Quelle (URL, Bibliotheksname, o.Ä.).
    local_path : str
        Relativer lokaler Pfad der Datei.
    data_type : {'raw','intermediate','processed','output'}
        Art der Datei in der Pipeline.
    rows : int, optional
        Anzahl Zeilen (für tabellarische Daten).
    size_kb : float, optional
        Grösse in Kilobyte (wird auf 1 Nachkommastelle gerundet).
    status : {'active','superseded','deleted'}, default 'active'
        Status des Eintrags.
    note : str, default ''
        Freitext-Kommentar.
    dataindex_path : str, optional
        Pfad zur ``dataindex.csv``. Wenn ``None``, wird im NB-Scope die
        globale Variable ``DATAINDEX`` gesucht (Rückwärtskompatibilität);
        Fallback ``"../sync/dataindex.csv"``.

    Return
    ------
    None. Schreibt nach ``dataindex.csv``.
    """
    import pandas as pd

    # dataindex_path auflösen
    if dataindex_path is None:
        # Versuche globale Variable DATAINDEX aus dem aufrufenden Scope
        import inspect
        caller_globals = inspect.stack()[1].frame.f_globals
        dataindex_path = caller_globals.get('DATAINDEX', '../sync/dataindex.csv')

    ts = datetime.utcnow().isoformat(timespec='seconds') + 'Z'

    if os.path.exists(dataindex_path):
        df_idx = pd.read_csv(dataindex_path)
        mask = (df_idx['filename'] == filename) & (df_idx['status'] == 'active')
        if mask.any():
            df_idx.loc[mask, 'status']         = 'superseded'
            df_idx.loc[mask, 'superseded_at']  = ts
    else:
        df_idx = pd.DataFrame(columns=DATAINDEX_COLUMNS)

    row = {
        'timestamp':      ts,
        'filename':       filename,
        'source_url':     source_url,
        'local_path':     local_path,
        'data_type':      data_type,
        'rows':           rows,
        'size_kb':        round(size_kb, 1) if size_kb else None,
        'status':         status,
        'superseded_at':  '',
        'note':           note,
    }
    pd.concat(
        [df_idx, pd.DataFrame([row])],
        ignore_index=True,
    ).to_csv(dataindex_path, index=False)

    print(f'  dataindex: {filename} [{status}]')
```

</details>


**Rohdaten laden:** Spot-Preise und Netzlast aus `data/raw/` einlesen;
Zeitfeatures (`hour`, `month`, `season`) berechnen.


In [9]:
# ── Rohdaten laden ────────────────────────────────────────────────────────────
df_prices = pd.read_csv(os.path.join(DIR_RAW, 'ch_spot_prices_raw.csv'))
df_load   = pd.read_csv(os.path.join(DIR_RAW, 'ch_netzlast_raw.csv'))
print(f'MODE       : {MODE}')
print(f'Preisdaten : {len(df_prices):,} Zeilen | {list(df_prices.columns)}')
print(f'Lastdaten  : {len(df_load):,} Zeilen  | {list(df_load.columns)}')


MODE       : data
Preisdaten : 29,084 Zeilen | ['timestamp', 'price_eur_mwh']
Lastdaten  : 29,070 Zeilen  | ['timestamp', 'load_gw']


---
## 1. Datenbereinigung Spot-Preise <a id='datenbereinigung-spot-preise_NB_02'></a>

[↑ Inhaltsverzeichnis](#toc_NB_02)

Clippt physikalisch unplausible Preise (< −500 / > 3 000 EUR/MWh), schliesst
Kurzlücken per linearer Interpolation (≤3h) mit Forward/Backward-Fill als
Fallback (≤6h) und ergänzt Zeitfeatures (`hour`, `month`, `season`).
Ergebnis: `ch_spot_prices_clean.csv` in `processed/`.


**🔎 Quellcode der importierten lib-Funktion**

`needs_rebuild` aus `lib.io_ops` — aufklappbar ist der Quellcode einsehbar.


In [10]:
show_source(needs_rebuild)


<details>
<summary>🔎 Quellcode: <code>needs_rebuild</code> (aus <code>lib/io_ops.py</code>)</summary>

```python
def needs_rebuild(filepath, min_rows, ds_key, force_reload=None):
    """True wenn Datei fehlt, zu wenige Zeilen, oder force_reload gesetzt.

    Pendant zu needs_download, aber zeilen-basiert — für Intermediate-CSV-
    Dateien, wo die Byte-Grösse wenig aussagt.

    Parameter
    ---------
    filepath : str
        Pfad zur zu prüfenden Datei.
    min_rows : int
        Minimale erwartete Anzahl Datenzeilen (Header zählt nicht).
    ds_key : str
        Key für den force_reload-Dict.
    force_reload : dict, optional
        Wie in needs_download — Fallback auf ``FORCE_RELOAD`` im Caller-Scope.

    Return
    ------
    bool
        True → muss neu erzeugt werden.
    """
    if force_reload is None:
        import inspect
        caller_globals = inspect.stack()[1].frame.f_globals
        force_reload = caller_globals.get('FORCE_RELOAD', {})

    if force_reload.get(ds_key, False):
        print(f'  FORCE_RELOAD={ds_key} → neu erzeugen')
        return True
    if not os.path.exists(filepath):
        return True
    try:
        n = sum(1 for _ in open(filepath)) - 1
        if n < min_rows:
            print(f'  Zu wenig Zeilen ({n} < {min_rows}) → neu erzeugen')
            return True
    except Exception:
        return True
    return False
```

</details>


In [11]:
# FORCE_RELOAD['clean'] = True → Zelle neu ausführen erzwingt Neuberechnung
CLEAN_FILE = os.path.join(DIR_PROCESSED, 'ch_spot_prices_clean.csv')

if needs_rebuild(CLEAN_FILE, 17000, 'clean'):
    # df_prices: Rohdaten aus vorheriger Zelle
    df_prices = pd.read_csv(os.path.join(DIR_RAW, 'ch_spot_prices_raw.csv'),
                             parse_dates=['timestamp'])
    # 1a: UTC normieren
    df_prices['timestamp'] = pd.to_datetime(df_prices['timestamp'], utc=True)
    df_prices = df_prices.sort_values('timestamp').reset_index(drop=True)
    # 1b: Vollständigen Stundenindex erzwingen
    full_idx = pd.date_range(df_prices['timestamp'].min(),
                              df_prices['timestamp'].max(), freq='1h', tz='UTC')
    df_prices = df_prices.set_index('timestamp').reindex(full_idx).reset_index()
    df_prices.rename(columns={'index': 'timestamp'}, inplace=True)
    print(f'1b – Fehlende Stunden: {df_prices["price_eur_mwh"].isna().sum()}')
    # 1c–1e: Interpolieren, Ausreisser, Zeitfeatures
    df_prices['price_eur_mwh'] = (df_prices['price_eur_mwh']
        .interpolate(method='linear', limit=3)
        .fillna(method='ffill', limit=6).fillna(method='bfill', limit=6))
    n_clip = ((df_prices['price_eur_mwh'] < -500) | (df_prices['price_eur_mwh'] > 3000)).sum()
    df_prices['price_eur_mwh'] = df_prices['price_eur_mwh'].clip(-500, 3000)
    print(f'1d – Ausreisser: {n_clip}')
    df_prices['hour']    = df_prices['timestamp'].dt.hour
    df_prices['month']   = df_prices['timestamp'].dt.month
    df_prices['weekday'] = df_prices['timestamp'].dt.dayofweek
    df_prices['season']  = df_prices['month'].map(
        {12:0,1:0,2:0,3:1,4:1,5:1,6:2,7:2,8:2,9:3,10:3,11:3})
    # 1f: Speichern
    df_prices.to_csv(CLEAN_FILE, index=False)
    kb = os.path.getsize(CLEAN_FILE) / 1024
    log_dataindex('ch_spot_prices_clean.csv', 'NB2:bereinigung', CLEAN_FILE, 'processed',
                  rows=len(df_prices), size_kb=kb, note=f'MODE={MODE}')
    print(f'Bereinigt → {CLEAN_FILE} | {len(df_prices):,} h | {kb:.0f} KB')
else:
    df_prices = pd.read_csv(CLEAN_FILE, parse_dates=['timestamp'])
    print(f'ch_spot_prices_clean vorhanden ({len(df_prices):,} Z) – Bereinigung übersprungen.\nFür ein erneutes Rebuild config.json force_reload.clean auf true setzen')

1b – Fehlende Stunden: 3
1d – Ausreisser: 0
  dataindex: ch_spot_prices_clean.csv [active]
Bereinigt → ../data\processed\ch_spot_prices_clean.csv | 29,087 h | 1197 KB


**Verifikation:** Shape, Nullwerte und Wertebereich der bereinigten Preise prüfen.


In [12]:
# ── Verifikation: Bereinigte Preise ─────────────────────────────────────────
print(f'Shape   : {df_prices.shape}')
print(f'Dtypes  : timestamp={df_prices["timestamp"].dtype}, '
      f'price={df_prices["price_eur_mwh"].dtype}')
print(f'Features: {[c for c in df_prices.columns if c not in ["timestamp","price_eur_mwh"]]}')
print(f'Nulls   : {df_prices.isnull().sum().sum()}')
df_prices.head(3)


Shape   : (29087, 6)
Dtypes  : timestamp=datetime64[ns, UTC], price=float64
Features: ['hour', 'month', 'weekday', 'season']
Nulls   : 0


,timestamp,price_eur_mwh,hour,month,weekday,season
0,2022-12-31 23:00:00+00:00,0.03,23,12,5,0
1,2023-01-01 00:00:00+00:00,-7.25,0,1,6,0
2,2023-01-01 01:00:00+00:00,-3.99,1,1,6,0


---
## Fazit <a id='fazit_NB_02'></a>

[↑ Inhaltsverzeichnis](#toc_NB_02)

Bereinigte Preisreihe ohne Duplikate und mit gefüllten Kurzlücken vorhanden.
Datenqualität: >99% Abdeckung pro Jahr, verbleibende `NaN` sind dokumentiert.
Die Reihe ist der Input für Dispatch-Simulation (NB03) und alle Kür-Analysen.


---
## Abschluss <a id='abschluss_NB_02'></a>

[↑ Inhaltsverzeichnis](#toc_NB_02)

Pflicht-Ausgabedateien auf Existenz und Mindestgrösse prüfen.
Fehler müssen vor NB03 behoben werden.

**🔎 Quellcode der importierten lib-Funktion**

Die Funktion `final_check` wird aus `lib/io_ops.py` importiert und
in der folgenden Zelle verwendet. Aufklappbar ist der Quellcode einsehbar.


In [13]:
show_source(final_check)

<details>
<summary>🔎 Quellcode: <code>final_check</code> (aus <code>lib/io_ops.py</code>)</summary>

```python
def final_check(nb_label, files=None, *, weiter_msg=None, fehler_msg=None,
                extras=None, show_dataindex=False,
                dataindex_path='../sync/dataindex.csv', width=60):
    """Standardisierte End-of-Notebook-Kontrolle für Pflicht- und Kür-NBs.

    Prüft Existenz und Mindestgrösse der angegebenen Output-Dateien,
    gibt formatiertes Resultat aus und liefert ``all_ok`` als Bool zurück.

    Parameter
    ---------
    nb_label : str
        Label des Notebooks im Output-Header, z.B. ``"NB01"``, ``"K_03"``.
    files : list of tuple, optional
        Zu prüfende Dateien als ``(path, label, min_bytes)``-Tuples.

        * ``min_bytes = 0`` → nur Existenz prüfen, Grösse nicht ausgeben
          (z.B. für PNG-Charts).
        * ``min_bytes > 0`` → zusätzlich Grösse prüfen und in KB/MB ausgeben
          (z.B. für CSV-Dateien).

        Bei ``files=None`` oder ``files=[]`` wird kein Check ausgeführt;
        die Funktion dient dann als reiner Status-Print (für Report-NBs
        ohne eigene Outputs wie K_00).
    weiter_msg : str, optional
        Nachricht für den Erfolgsfall, z.B. ``"NB02 Daten Bereinigung"``.
        Default: ``"nächstes Notebook"``.
    fehler_msg : str, optional
        Nachricht für den Fehlerfall (Kurzform, ohne "Fehler beheben vor").
        Default: identisch mit ``weiter_msg``.
    extras : list of str, optional
        Zusätzliche Print-Zeilen zwischen Datei-Check und Weiter-/Fehler-Hinweis.
        Sinnvoll für Kür-Hinweise oder Kontext.
    show_dataindex : bool, default False
        Wenn True, wird der aktive Auszug aus ``../sync/dataindex.csv`` ausgegeben.
        Typisch für NB01.
    dataindex_path : str, default '../sync/dataindex.csv'
        Pfad zur dataindex.csv (für ``show_dataindex=True``).
    width : int, default 60
        Breite der Trennlinie aus ``=``-Zeichen.

    Return
    ------
    bool
        ``True`` wenn alle Files existieren und Mindestgrösse erfüllen,
        ``False`` sonst. Bei ``files=None``/leer immer ``True``.
    """
    print(f'{nb_label} – Abschlusskontrolle')
    print('=' * width)

    all_ok = True

    if files:
        for path, label, min_bytes in files:
            exists = os.path.exists(path)
            size = os.path.getsize(path) if exists else 0
            ok = exists and size >= min_bytes

            if min_bytes > 0:
                size_str = _format_size(size) if exists else '   FEHLT'
                print(f'  {"✅" if ok else "❌"}  {label:<45} {size_str}')
            else:
                print(f'  {"✅" if ok else "❌"}  {label}')

            if not ok:
                all_ok = False

    if extras:
        if files:
            print()
        for line in extras:
            print(line)

    if show_dataindex and os.path.exists(dataindex_path):
        import pandas as pd
        df_idx = pd.read_csv(dataindex_path)
        active = df_idx[df_idx['status'] == 'active']
        print(f'\ndataindex.csv: {len(df_idx)} Einträge total, {len(active)} active')
        print(active[['filename', 'data_type', 'rows', 'size_kb', 'timestamp']]
              .to_string(index=False))

    print()
    weiter = weiter_msg or 'nächstes Notebook'
    fehler = fehler_msg or weiter
    if all_ok:
        print(f'→ Weiter mit {weiter}.')
    else:
        print(f'→ Fehler beheben vor {fehler}.')

    return all_ok
```

</details>


In [14]:
# ── Abschlusskontrolle NB02 ──────────────────────────────────────────────────
final_check(
    'NB02',
    files=[
        (os.path.join(DIR_PROCESSED, 'ch_spot_prices_clean.csv'),
         'Bereinigte Spot-Preise (ch_spot_prices_clean.csv)', 100_000),
    ],
    weiter_msg='NB03 Daten Analyse',
    fehler_msg='NB03',
)

NB02 – Abschlusskontrolle
  ✅  Bereinigte Spot-Preise (ch_spot_prices_clean.csv)     1.2 MB

→ Weiter mit NB03 Daten Analyse.


True

| [← NB01 Daten Laden](01_Daten_Laden.ipynb) | [↑ Übersicht](../organisation/O_01_Project_Overview.ipynb) | [NB03 Daten Analyse →](03_Daten_Analyse.ipynb) |
|:---|:---:|---:|